# Section 6 – Expose the models with FastAPI
**STUDENT notebook: complete the TODO cells**

| | |
|---|---|
| Connects to | Topic 2 – API design, Topic 3 – FastAPI |
| Estimated time | 70 minutes |
| Produces | `app/main.py` (the API) and a passing test suite |

**Goal:** Build the real API: list and read characters, predict from stats, one standard error format, automatic docs, tests with TestClient and a live server.

### How to work in this notebook
- 🟦 **Worked sample** cells run as they are. Read them, run them, change them.
- 🟨 **Your turn** cells contain `# TODO` and `...`. Replace `...` with your code.
- ✅ **Check** cells print `PASS` / `FAIL`. Do not move on while you see a `FAIL`.
- Stuck? Open the same file in the `solution/` folder, but try first.

## Step 0 – Setup

In [ ]:
import importlib.util
import subprocess
import sys

# 1) Make sure the libraries exist in THIS kernel (installs only what is missing)
REQUIRED = {"requests": "requests", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn", "matplotlib": "matplotlib", "pillow": "PIL",
            "joblib": "joblib", "fastapi": "fastapi", "uvicorn": "uvicorn",
            "httpx": "httpx", "pydantic": "pydantic"}
missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing, "-> restart the kernel if an import below fails")
else:
    print("All dependencies are available")

# 2) Corporate networks with HTTPS inspection: use the Windows certificate store if available
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore active (Windows certificate store)")
except ImportError:
    pass

# 3) Lab folders (this notebook lives in student/ or solution/, so the lab root is one level up)
from pathlib import Path
import json

ROOT = Path.cwd().parent
DATA, MODELS, OUT = ROOT / "data", ROOT / "models", ROOT / "outputs"
for folder in (DATA, MODELS, OUT):
    folder.mkdir(exist_ok=True)


def check(name, ok, detail=""):
    """Self-check helper: prints PASS / FAIL."""
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


print("Lab root:", ROOT)

## The contract we implement
From Topic 2, now with our two models behind it:

| Route | Purpose | Success | Errors |
|---|---|---|---|
| `GET /` | health check | 200 | |
| `GET /v1/characters` | list (paginated, filters `name`, `legendary`) | 200 | 422 |
| `GET /v1/characters/{id}` | one character + real power + **prediction** | 200 | 404 |
| `POST /v1/predictions` | predict from 8 stats | 200 | 422 |

All errors share **one shape** (Topic 2):
```json
{"error": {"code": "CHARACTER_NOT_FOUND", "message": "...", "details": null}}
```

Requirement: finish **Section 5** first (the `app/` package must exist). If not, copy `solution/app/` into `student/app/`.

## Step 1 – One standard error format
🟦 **Worked sample**: a mini app (not our real one) with a custom exception and a handler.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient


class DemoError(Exception):
    def __init__(self, status, code, message):
        self.status, self.code, self.message = status, code, message


demo = FastAPI()


@demo.exception_handler(DemoError)
async def demo_error_handler(request, exc):
    return JSONResponse(status_code=exc.status,
                        content={"error": {"code": exc.code, "message": exc.message, "details": None}})


@demo.get("/boom")
def boom():
    raise DemoError(404, "NOT_HERE", "Nothing here")


r = TestClient(demo).get("/boom")
print(r.status_code, r.json())

## Step 2 – Write `app/main.py`
Pieces to notice:

- `lifespan`: code that runs **once at startup**: load the models and the table (Topic 5 idea).
- `state`: a dict that keeps those objects for all requests.
- `response_model=...`: FastAPI checks that the answer follows the contract.
- Three exception handlers: our `ApiError`, validation errors (422) and framework errors (unknown URL, wrong method).

🟨 You complete the three business routes.

In [ ]:
%%writefile app/main.py
"""Character Card Studio API: validated, documented and testable."""
from contextlib import asynccontextmanager
from typing import Optional

import pandas as pd
from fastapi import FastAPI, Query
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from starlette.exceptions import HTTPException as StarletteHTTPException

from .config import DATA_DIR
from .model_service import ModelService
from .schemas import CharacterDetail, CharacterStats, PredictionResponse

state = {}          # objects shared by every request (filled at startup)


@asynccontextmanager
async def lifespan(app: FastAPI):
    state["service"] = ModelService()
    state["df"] = pd.read_csv(DATA_DIR / "pokemon_clean.csv")
    yield
    state.clear()


app = FastAPI(
    title="Character Card Studio API",
    version="1.0.0",
    description="Characters, ML predictions (power + rarity) and collectible cards.",
    lifespan=lifespan,
)


# ---------- one standard error format ----------
class ApiError(Exception):
    def __init__(self, status: int, code: str, message: str):
        self.status, self.code, self.message = status, code, message


def error_response(status, code, message, details=None):
    return JSONResponse(status_code=status, content={"error": {"code": code, "message": message, "details": details}})


@app.exception_handler(ApiError)
async def api_error_handler(request, exc: ApiError):
    return error_response(exc.status, exc.code, exc.message)


@app.exception_handler(RequestValidationError)
async def validation_handler(request, exc: RequestValidationError):
    details = [{"field": ".".join(str(p) for p in e["loc"][1:]), "message": e["msg"]} for e in exc.errors()]
    return error_response(422, "INVALID_INPUT", "The request data is not valid", details)


@app.exception_handler(StarletteHTTPException)
async def http_error_handler(request, exc: StarletteHTTPException):
    return error_response(exc.status_code, "HTTP_ERROR", str(exc.detail))


def get_row_or_404(character_id: int):
    match = state["df"][state["df"]["id"] == character_id]
    if match.empty:
        raise ApiError(404, "CHARACTER_NOT_FOUND", f"Character {character_id} does not exist")
    return match.iloc[0]


# ---------- routes ----------
@app.get("/", tags=["health"])
def health():
    return {"status": "ok", "model_version": state["service"].version, "characters": int(len(state["df"]))}


@app.get("/v1/characters", tags=["characters"])
def list_characters(
    page: int = Query(1, ge=1),
    limit: int = Query(10, ge=1, le=50),
    name: Optional[str] = Query(None, description="part of the name"),
    legendary: Optional[bool] = Query(None, description="only legendary (true) or only common (false)"),
):
    df = state["df"]
    # TODO: apply the optional filters: name (contains, lower case) and legendary (compare with 1 / 0)
    # HINT: df[df["name"].str.contains(name.lower(), regex=False)]
    ...  # <- write your code here
    # TODO: take only the requested page (see Topic 2: start = (page - 1) * limit)
    ...  # <- write your code here
    items = page_df[["id", "name", "type1", "type2", "generation", "is_legendary"]].to_dict(orient="records")
    for item in items:
        item["is_legendary"] = bool(item["is_legendary"])
    return {"items": items, "page": page, "limit": limit, "total": int(len(df))}


@app.get("/v1/characters/{character_id}", response_model=CharacterDetail, tags=["characters"])
def get_character(character_id: int):
    row = get_row_or_404(character_id)
    service = state["service"]
    stats = {f: int(row[f]) for f in service.features}
    # TODO: ask the model service for the prediction of these stats
    ...  # <- write your code here
    return {
        "id": int(row["id"]), "name": row["name"], "type1": row["type1"], "type2": row["type2"],
        "generation": int(row["generation"]), "artwork_url": row["artwork_url"],
        "stats": stats, "actual_power": int(row["base_experience"]),
        "is_legendary": bool(row["is_legendary"]), "prediction": prediction,
    }


@app.post("/v1/predictions", response_model=PredictionResponse, tags=["predictions"])
def create_prediction(stats: CharacterStats):
    # TODO: FastAPI already validated `stats`; convert it to a dict and call the model service
    # HINT: stats.model_dump()
    ...  # <- write your code here

## Step 3 – Import the app and list its routes

In [ ]:
import sys
import time
import threading
import requests
import uvicorn
import pandas as pd
from fastapi.testclient import TestClient

sys.path.insert(0, str(Path.cwd()))
for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[name]

from app.main import app

print("Routes:")
for route in app.routes:
    methods = ",".join(sorted(getattr(route, "methods", None) or []))
    if methods and route.path not in ("/openapi.json", "/docs", "/docs/oauth2-redirect", "/redoc"):
        print(f"  {methods:<6} {route.path}")

## Step 4 – Test it without a server
`with TestClient(app) as client:` also runs the startup code (`lifespan`), so the models are loaded.
Remember Topic 2's rule: **one test = one principle**.

In [ ]:
RESULTS = []


def test(name, ok, detail=""):
    RESULTS.append(ok)
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


client_cm = TestClient(app)
client = client_cm.__enter__()          # starts the app (runs lifespan)

r = client.get("/")
test("health -> 200", r.status_code == 200, r.json())

r = client.get("/v1/characters", params={"limit": 3})
body = r.json()
test("list -> 200 with the pagination envelope", r.status_code == 200 and {"items", "page", "limit", "total"} <= set(body))
test("page size is respected", len(body["items"]) == 3)

🟨 **Your turn.** Write four more tests with `test(name, condition)`:

1. Filter `legendary=true` returns only legendary characters.
2. Filter `name=pika` finds Pikachu.
3. `GET /v1/characters/999999` returns **404** and the standard error shape (`error.code == "CHARACTER_NOT_FOUND"`).
4. `POST /v1/predictions` with `attack: 0` returns **422** and `error.code == "INVALID_INPUT"`.

In [ ]:
# TODO: test 1: only legendary characters
...  # <- write your code here

# TODO: test 2: name filter finds pikachu
...  # <- write your code here

# TODO: test 3: unknown id -> 404 + standard error shape
...  # <- write your code here

# TODO: test 4: attack = 0 -> 422 INVALID_INPUT (send a full valid body except attack)
...  # <- write your code here

In [ ]:
# One character with its prediction: how far is the model from the real value?
r = client.get("/v1/characters/25")
detail = r.json()
test("character 25 is pikachu", detail["name"] == "pikachu")
print(f'  actual power: {detail["actual_power"]} | predicted: {detail["prediction"]["predicted_power"]} '
      f'| rarity: {detail["prediction"]["rarity"]}')

mewtwo = {"hp": 106, "attack": 110, "defense": 90, "special_attack": 154, "special_defense": 90,
          "speed": 130, "height": 20, "weight": 1220}
r = client.post("/v1/predictions", json=mewtwo)
test("POST /v1/predictions -> 200 and legendary", r.status_code == 200 and r.json()["rarity"] == "legendary", r.json())
test("unknown URL -> 404 standard error", client.get("/nope").json()["error"]["code"] == "HTTP_ERROR")
test("GET on a POST route -> 405", client.get("/v1/predictions").status_code == 405)
print(f"\n{sum(RESULTS)}/{len(RESULTS)} tests passed")

## Step 5 – The documentation is already written
FastAPI built an OpenAPI description from our code and models.

In [ ]:
spec = client.get("/openapi.json").json()
print(spec["info"]["title"], spec["info"]["version"])
for path, methods in spec["paths"].items():
    print(f"  {path:<32} {', '.join(m.upper() for m in methods)}")
print("\nSchemas:", list(spec["components"]["schemas"]))
print(json.dumps(spec["components"]["schemas"]["CharacterStats"]["properties"]["attack"], indent=2))

## Step 6 – Start a real server and call it like any other client
`uvicorn` listens on a port. We run it in a background thread so the notebook keeps working, and call it with
`requests` exactly as in Topic 1. Open `http://127.0.0.1:8770/docs` in your browser while it runs and try the endpoints.

In [ ]:
client_cm.__exit__(None, None, None)    # close the TestClient

PORT = 8770
BASE_URL = f"http://127.0.0.1:{PORT}"
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

for _ in range(100):
    try:
        requests.get(BASE_URL, timeout=0.5)
        break
    except requests.exceptions.RequestException:
        time.sleep(0.1)
print("Server running at", BASE_URL, "| docs:", BASE_URL + "/docs")

In [ ]:
r = requests.get(f"{BASE_URL}/v1/characters/150", timeout=10)
d = r.json()
print(r.status_code, r.headers["server"], "|", d["name"], "| actual:", d["actual_power"],
      "| predicted:", d["prediction"]["predicted_power"], "| rarity:", d["prediction"]["rarity"])

r = requests.post(f"{BASE_URL}/v1/predictions", json={**mewtwo, "attack": "very strong"}, timeout=10)
print(r.status_code, json.dumps(r.json(), indent=1))

In [ ]:
server.should_exit = True
thread.join(timeout=5)
print("server stopped:", not thread.is_alive())

## Step 7 – Run it like a real project
Open a terminal **in the `solution/` or `student/` folder** (where `app/` lives):

```bash
python -m uvicorn app.main:app --reload --port 8000
```
Then open <http://127.0.0.1:8000/docs> → *POST /v1/predictions* → **Try it out**.

| Symptom | Fix |
|---|---|
| `Could not import module "app.main"` | run the command from the folder that contains `app/` |
| `Address already in use` | use another port: `--port 8001` |
| `FileNotFoundError ... regressor.joblib` | finish Sections 3 and 4, or copy `solution_artifacts/models/` into `models/` |

## Questions
1. Why do we return `404` for an unknown character and `422` for `attack = 0`? Who made the mistake in each case?
2. What would happen to response time if `ModelService()` were created inside each route?
3. What does `response_model=CharacterDetail` protect the API consumer from?
4. Compare `actual_power` and `predicted_power` for 5 characters. Where does the model fail?

### Extra challenge
Add `GET /v1/characters/{id}/prediction` that returns **only** the prediction part, with the right response model.

**Next:** Section 7 – draw the card and share it.